In [1]:
# 필요한 라이브러리 설치
!pip install transformers datasets torch -q

print("✅ 설치 완료")

✅ 설치 완료


In [2]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

print(f"✅ GPU 사용 가능: {torch.cuda.is_available()}")
print(f"   GPU 이름: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

✅ GPU 사용 가능: True
   GPU 이름: Tesla T4


In [3]:
from google.colab import files
uploaded = files.upload()
print("✅ 파일 업로드 완료")

MessageError: RangeError: Maximum call stack size exceeded.

In [4]:
df = pd.read_csv(
    'SMSSpamCollection',
    sep='\t',
    header=None,
    names=['label', 'message'],
    encoding='utf-8'
)
df['label_num'] = (df['label'] == 'spam').astype(int)

print(f"✅ 데이터 로드 완료: {df.shape}")
print(df['label'].value_counts())

FileNotFoundError: [Errno 2] No such file or directory: 'SMSSpamCollection'

In [5]:
from google.colab import files # Keep this import for potential future use or other file operations
import io # Keep this import for potential future use
import pandas as pd # Ensure pandas is imported for DataFrame operations

# The previous call to files.upload() failed with a MessageError: RangeError: Maximum call stack size exceeded.
# This error typically indicates an issue with Colab's file upload mechanism.
# To bypass this, please manually upload your 'SMSSpamCollection' file (or the correct file name)
# using the 'Files' icon (folder icon) on the left sidebar in Google Colab.

# Assuming the file 'SMSSpamCollection' has been manually uploaded to the Colab environment
# Commenting out the problematic upload lines:
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]

# Use a placeholder filename that matches the expected data file
filename = 'SMSSpamCollection' # Please ensure this matches the name of your manually uploaded file

df = pd.read_csv(
    filename, # Read directly from the file system after manual upload
    # io.BytesIO(uploaded[filename]), # This line is no longer needed if reading directly from file system
    sep='\t',
    header=None,
    names=['label', 'message'],
    encoding='utf-8'
)
df['label_num'] = (df['label'] == 'spam').astype(int)

print(f"✅ 데이터 로드 완료: {df.shape}")
print(df['label'].value_counts())

MessageError: RangeError: Maximum call stack size exceeded.

In [6]:
df = pd.read_csv(
    'SMSSpamCollection',
    sep='\t',
    header=None,
    names=['label', 'message'],
    encoding='utf-8'
)
df['label_num'] = (df['label'] == 'spam').astype(int)

print(f"✅ 데이터 로드 완료: {df.shape}")
print(df['label'].value_counts())

✅ 데이터 로드 완료: (5572, 3)
label
ham     4825
spam     747
Name: count, dtype: int64


In [7]:
# Train/Test 분리
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['message'].tolist(),
    df['label_num'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label_num']
)

# BERT 토크나이저 로드
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# 데이터셋 클래스 정의
class SpamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(texts, truncation=True, padding=True,
                                   max_length=max_len, return_tensors='pt')
        self.labels = torch.tensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_dataset = SpamDataset(train_texts, train_labels, tokenizer)
test_dataset  = SpamDataset(test_texts,  test_labels,  tokenizer)

print(f"✅ 데이터셋 준비 완료")
print(f"   Train: {len(train_dataset)}, Test: {len(test_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ 데이터셋 준비 완료
   Train: 4457, Test: 1115


In [8]:
# BERT 모델 로드
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc}

# 학습 설정
training_args = TrainingArguments(
    output_dir='./bert_spam',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("✅ 학습 시작!")
trainer.train()
print("✅ 학습 완료!")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will 

✅ 학습 시작!


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

✅ 학습 완료!


In [9]:
# 최종 평가
predictions = trainer.predict(test_dataset)
preds = predictions.predictions.argmax(-1)

print("="*50)
print("📊 BERT 최종 성능")
print("="*50)
print(classification_report(test_labels, preds, target_names=['ham', 'spam']))
print(f"\n🏆 BERT Accuracy: {accuracy_score(test_labels, preds):.4f}")

# 1단계 vs 2단계 비교
print("\n" + "="*50)
print("📈 모델 성능 비교")
print("="*50)
print(f"Logistic Regression : 97.13%")
print(f"Random Forest       : 97.49%")
print(f"XGBoost             : 97.22%")
print(f"BERT (Fine-tuned)   : {accuracy_score(test_labels, preds)*100:.2f}% ✅ Best!")

📊 BERT 최종 성능
              precision    recall  f1-score   support

         ham       0.99      0.99      0.99       966
        spam       0.96      0.96      0.96       149

    accuracy                           0.99      1115
   macro avg       0.98      0.98      0.98      1115
weighted avg       0.99      0.99      0.99      1115


🏆 BERT Accuracy: 0.9892

📈 모델 성능 비교
Logistic Regression : 97.13%
Random Forest       : 97.49%
XGBoost             : 97.22%
BERT (Fine-tuned)   : 98.92% ✅ Best!
